In [33]:
import os

for dirname, subdirs, filenames in os.walk('/kaggle/input'):
    # Print directories
    for subdir in subdirs:
        print(os.path.join(dirname, subdir))
    # Print only .pth files
    for filename in filenames:
        if filename.endswith('.pth'):
            print(os.path.join(dirname, filename))

/kaggle/input/datasets
/kaggle/input/datasets/anaylg
/kaggle/input/datasets/anaylg/oadn-dataset
/kaggle/input/datasets/anaylg/oadn-dataset/OADN
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/Implementation Code
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/Preprocessed Testing Data
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/Preprocessed Training Data
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/RAF-DB
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/Preprocessed Testing Data/test_landmarks_confidencescores.pth
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/Preprocessed Testing Data/test_24_interest_points.pth
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/Preprocessed Training Data/train_landmarks_confidencescores.pth
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/Preprocessed Training Data/train_24_interest_points.pth
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/RAF-DB/2
/kaggle/input/datasets/anaylg/oadn-dataset/OADN/RAF-DB/raf-db-dataset
/kaggle/input/datasets/anaylg

In [34]:
import numpy as np
import pandas as pd
import os
import torch
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torchvision.models as models
from PIL import Image

In [35]:
"""
This script defines the OADN backbone, which is a modified ResNet-50.

The modifications from the standard ResNet-50 are:
1. The final average pooling layer and fully connected layer are
   removed, since OADN builds its own classification heads through
   its two branches.
2. The stride of the first block in layer4 (conv5_x in ResNet
   notation, referred to as conv4_1 in the OADN paper) is changed
   from 2 to 1. This keeps the output feature map at 14x14 spatial
   resolution instead of the default 7x7.

The result is that for a 224x224x3 input image, the backbone
produces a 14x14x2048 feature map. This feature map is shared
between the landmark-guided attention branch and the facial
region branch of the OADN model.
"""

class OADNBackbone(nn.Module):
    """
    Description:
        Modified ResNet-50 backbone for the OADN model. Loads a
        pretrained ResNet-50, removes the average pooling and
        fully connected layers, and changes the stride of layer4's
        first block from 2 to 1 so the output feature map is
        14x14x2048 instead of 7x7x2048.

    Input:
        x: torch.Tensor of shape (batch_size, 3, 224, 224)
        A batch of RGB images normalized with ImageNet statistics.

    Output:
        torch.Tensor of shape (batch_size, 2048, 14, 14)
        The feature map extracted from the modified ResNet-50.
    """

    def __init__(self):
        super(OADNBackbone, self).__init__()

        # Load pretrained ResNet-50
        resnet = models.resnet50(pretrained=True)

        # Modify the stride of layer4's first block from 2 to 1
        # The stride is applied in two places:
        # 1. The 3x3 convolution in the middle of the bottleneck block
        resnet.layer4[0].conv2.stride = (1, 1)
        # 2. The downsample shortcut connection
        resnet.layer4[0].downsample[0].stride = (1, 1)

        # Keep all layers except avgpool and fc
        self.features = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4
        )

    def forward(self, x):
        return self.features(x)


In [36]:
"""
This script defines a custom PyTorch Dataset class for the RAF-DB
dataset that returns three things per image: the transformed image
tensor, the expression label, and the 24 attention maps.

The standard ImageFolder dataset only returns images and labels.
We need the attention maps as well because the OADN model's LAB
branch requires them during both training and testing.

The class loads the precomputed 24 interest points and their
confidence scores from a .pth file. During __getitem__, for each
image it:
1. Loads and transforms the image (resize, flip, normalize)
2. Looks up the precomputed interest points and scores
3. Applies the confidence threshold T to determine visibility
4. Generates 24 Gaussian attention heatmaps of size 14x14
5. Returns the image tensor, label, and attention maps tensor

The attention maps are generated on the fly rather than precomputed
so that the threshold T can be changed without re-running a
preprocessing script.
"""

def generate_attention_maps(landmarks, scores, T=0.6, sigma=4, map_size=14, image_size=224):
    """
    Description:
        Generates 24 Gaussian attention heatmaps from interest points.
        Points with confidence below T get all-zero heatmaps.
        Points with confidence >= T get a 2D Gaussian centered at
        their location, scaled to feature map space.

    Input:
        landmarks: numpy array of shape (24, 2)
            The x,y coordinates of interest points in image space.
        scores: numpy array of shape (24,)
            Confidence scores for each interest point.
        T: float, default 0.6
            Confidence threshold. Points below this are occluded.
        sigma: float, default 4
            Standard deviation of the Gaussian.
        map_size: int, default 14
            Spatial size of the output heatmaps.
        image_size: int, default 224
            Spatial size of the input images.

    Output:
        attention_maps: torch.Tensor of shape (24, map_size, map_size)
            The Gaussian attention heatmaps.
    """
    num_points = landmarks.shape[0]
    attention_maps = np.zeros((num_points, map_size, map_size), dtype=np.float32)
    scale = map_size / image_size
    grid_y, grid_x = np.mgrid[0:map_size, 0:map_size]

    for i in range(num_points):
        if scores[i] < T:
            continue
        cx = landmarks[i, 0] * scale
        cy = landmarks[i, 1] * scale
        gaussian = np.exp(-((grid_x - cx) ** 2 + (grid_y - cy) ** 2) / (2 * sigma ** 2))
        attention_maps[i] = gaussian

    return torch.from_numpy(attention_maps)


class RAFDataset(Dataset):
    """
    Description:
        Custom Dataset for RAF-DB that returns image tensors,
        expression labels, and attention maps for the OADN model.

    Input (to __init__):
        image_dir: String
            Path to the image folder (e.g., .../DATASET/train).
        interest_points_path: String
            Path to the .pth file containing precomputed 24
            interest points and confidence scores per image.
        transform: torchvision.transforms.Compose
            Image transformations to apply.
        T: float, default 0.6
            Confidence threshold for attention map generation.

    Output (from __getitem__):
        image: torch.Tensor of shape (3, 224, 224)
            The transformed image.
        label: int
            The expression class label (0-6).
        attention_maps: torch.Tensor of shape (24, 14, 14)
            The Gaussian attention heatmaps.
    """

    def __init__(self, image_dir, interest_points_path, transform, T=0.6):
        self.image_dir = image_dir
        self.transform = transform
        self.T = T

        # Load precomputed interest points
        self.interest_points = torch.load(interest_points_path, weights_only=False)

        # Build list of (image_path, label, dict_key) for all images
        self.samples = []
        class_folders = sorted(os.listdir(image_dir))
        for class_idx, class_folder in enumerate(class_folders):
            class_path = os.path.join(image_dir, class_folder)
            if not os.path.isdir(class_path):
                continue
            for img_name in sorted(os.listdir(class_path)):
                img_path = os.path.join(class_path, img_name)
                dict_key = f"{class_folder}/{img_name}"
                self.samples.append((img_path, class_idx, dict_key))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        img_path, label, dict_key = self.samples[index]

        # Load and transform the image
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)

        # Look up precomputed interest points and generate attention maps
        if dict_key in self.interest_points:
            landmarks = self.interest_points[dict_key]["landmarks"]
            scores = self.interest_points[dict_key]["scores"]
            attention_maps = generate_attention_maps(landmarks, scores, T=self.T)
        else:
            # If landmarks were not found (failed during preprocessing),
            # use all-zero attention maps as fallback
            attention_maps = torch.zeros(24, 14, 14)

        return image, label, attention_maps

In [37]:
"""
This script defines the Landmark-guided Attention Branch (LAB)
of the OADN model.

The LAB takes the feature map F from the backbone and 24 attention
maps generated from facial landmarks. It performs the following:

1. Multiplies F element-wise with each of the 24 attention maps,
   producing 24 landmark-guided feature maps (Equation 2 in paper)
2. Applies global average pooling to each, giving 24 vectors of
   dimension 2048
3. Max pools across the 24 vectors into one 2048-d vector
4. Passes through FC (2048 -> 256) with ReLU activation
5. Passes through FC (256 -> 7) to produce class predictions

The output is raw logits for 7 expression classes. Softmax is
not applied here because PyTorch's CrossEntropyLoss handles it
internally.
"""

class LandmarkAttentionBranch(nn.Module):
    """
    Description:
        Implements the landmark-guided attention branch of OADN.
        Uses attention maps derived from facial landmarks to guide
        the network to focus on non-occluded facial regions.

    Input:
        features: torch.Tensor of shape (batch_size, 2048, 14, 14)
            Feature maps from the OADN backbone.
        attention_maps: torch.Tensor of shape (batch_size, 24, 14, 14)
            Gaussian attention heatmaps for the 24 interest points.
            Occluded points have all-zero heatmaps.

    Output:
        logits: torch.Tensor of shape (batch_size, 7)
            Raw prediction scores for 7 expression classes.
    """

    def __init__(self, num_classes=7):
        super(LandmarkAttentionBranch, self).__init__()
        self.fc1 = nn.Linear(2048, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, features, attention_maps):
        batch_size = features.shape[0]
        num_points = attention_maps.shape[1]

        # Step 1: Multiply features with each attention map
        # features: (batch_size, 2048, 14, 14)
        # attention_maps: (batch_size, 24, 14, 14)
        # Unsqueeze attention_maps to (batch_size, 24, 1, 14, 14) so it
        # broadcasts across the 2048 channels
        # Unsqueeze features to (batch_size, 1, 2048, 14, 14) so it
        # broadcasts across the 24 attention maps
        weighted = features.unsqueeze(1) * attention_maps.unsqueeze(2)
        # weighted shape: (batch_size, 24, 2048, 14, 14)

        # Step 2: Global average pooling over the 14x14 spatial dimensions
        pooled = weighted.mean(dim=[3, 4])
        # pooled shape: (batch_size, 24, 2048)

        # Step 3: Max pool across the 24 landmark-guided features
        fused, _ = pooled.max(dim=1)
        # fused shape: (batch_size, 2048)

        # Step 4: FC 2048 -> 256 with ReLU
        out = self.relu(self.fc1(fused))

        # Step 5: FC 256 -> 7
        logits = self.fc2(out)

        return logits

In [38]:
"""
This script defines the Facial Region Branch (FRB) of the OADN model.

The FRB takes the feature map F from the backbone and divides it
into K non-overlapping spatial blocks. Each block is independently
pooled and classified, so that even if some facial regions are
occluded, the remaining regions can still predict the expression.

The steps are:
1. Divide the 14x14 feature map into K blocks (default K=4,
   giving a 2x2 grid of 7x7 blocks)
2. Apply global average pooling to each block, producing K
   vectors of dimension 2048
3. Each vector passes through its own FC (2048 -> 256) with ReLU
4. Each vector passes through its own FC (256 -> 7) to predict
   the expression independently

Each region produces its own prediction and its own loss during
training. This adds more supervision and acts as a regularizer
to reduce overfitting, as described in Section 3.2 of the paper.
"""

import torch
import torch.nn as nn


class FacialRegionBranch(nn.Module):
    """
    Description:
        Implements the facial region branch of OADN. Divides the
        backbone feature map into non-overlapping blocks and trains
        an independent expression classifier on each block.

    Input:
        features: torch.Tensor of shape (batch_size, 2048, 14, 14)
            Feature maps from the OADN backbone.

    Output:
        region_logits: list of K torch.Tensors, each of shape
            (batch_size, 7). Each tensor is the raw prediction
            scores from one facial region's classifier.
    """

    def __init__(self, num_classes=7, grid_rows=2, grid_cols=2):
        super(FacialRegionBranch, self).__init__()

        self.grid_rows = grid_rows
        self.grid_cols = grid_cols
        K = grid_rows * grid_cols

        # Each region gets its own pair of FC layers
        self.fc1_layers = nn.ModuleList([nn.Linear(2048, 256) for _ in range(K)])
        self.fc2_layers = nn.ModuleList([nn.Linear(256, num_classes) for _ in range(K)])
        self.relu = nn.ReLU()

    def forward(self, features):
        batch_size, channels, height, width = features.shape

        # Compute the size of each block
        block_h = height // self.grid_rows
        block_w = width // self.grid_cols

        region_logits = []
        region_idx = 0

        for i in range(self.grid_rows):
            for j in range(self.grid_cols):
                # Extract the (i, j) block from the feature map
                block = features[:, :,
                                 i * block_h : (i + 1) * block_h,
                                 j * block_w : (j + 1) * block_w]

                # Global average pooling over the spatial dimensions
                pooled = block.mean(dim=[2, 3])

                # FC 2048 -> 256 with ReLU
                out = self.relu(self.fc1_layers[region_idx](pooled))

                # FC 256 -> 7
                logits = self.fc2_layers[region_idx](out)

                region_logits.append(logits)
                region_idx += 1

        return region_logits

In [39]:
"""
This script defines the OADN (Occlusion-Adaptive Deep Network) parent
model class that combines the backbone, landmark-guided attention
branch, and facial region branch into a single model.

The forward pass:
1. Passes the input image through the modified ResNet-50 backbone
   to get a 14x14x2048 feature map
2. Feeds the feature map and attention maps to the LAB branch
3. Feeds the feature map to the FRB branch
4. Returns predictions from both branches

During training, each branch's predictions are used to compute
separate losses which are combined with a weighted sum (Equation 5).
During testing, the predictions from both branches are averaged
to get the final prediction (Section 4.2 of the paper).
"""

class OADN(nn.Module):
    """
    Description:
        The complete OADN model. Contains the shared ResNet-50
        backbone and both classification branches. The backbone
        feature map is shared between the two branches.

    Input:
        images: torch.Tensor of shape (batch_size, 3, 224, 224)
            Batch of normalized RGB face images.
        attention_maps: torch.Tensor of shape (batch_size, 24, 14, 14)
            Gaussian attention heatmaps for the 24 interest points.

    Output:
        lab_logits: torch.Tensor of shape (batch_size, num_classes)
            Predictions from the landmark-guided attention branch.
        frb_logits: list of K torch.Tensors, each of shape
            (batch_size, num_classes)
            Predictions from each region classifier in the facial
            region branch.
    """

    def __init__(self, num_classes=7, grid_rows=2, grid_cols=2):
        super(OADN, self).__init__()
        self.backbone = OADNBackbone()
        self.lab = LandmarkAttentionBranch(num_classes=num_classes)
        self.frb = FacialRegionBranch(num_classes=num_classes,
                                      grid_rows=grid_rows,
                                      grid_cols=grid_cols)

    def forward(self, images, attention_maps):
        features = self.backbone(images)
        lab_logits = self.lab(features, attention_maps)
        frb_logits = self.frb(features)
        return lab_logits, frb_logits

In [40]:
"""
This script handles training and evaluation of the OADN model
on the RAF-DB dataset.

Training follows the paper's specifications:
- Optimizer: SGD with momentum 0.9, weight decay 0.0005
- Learning rate: starts at 0.1, reduced by 10x after 20 epochs
- Batch size: 128
- Total epochs: 60
- Data augmentation: random horizontal flip only
- Loss: weighted combination of LAB and FRB cross-entropy losses
  total_loss = lambda * LAB_loss + (1 - lambda) * FRB_loss

Testing follows Section 4.2 of the paper:
- Prediction scores from LAB and FRB branches are averaged
- The class with the highest averaged score is the final prediction
- Total accuracy and per-class confusion matrix are reported

The script first runs one full training with the paper's best
hyperparameters (T=0.6, K=4, lambda=0.6) to verify the pipeline.
An ablation sweep function is provided to tune one hyperparameter
at a time while keeping the others fixed.
"""

def train_one_epoch(model, train_loader, criterion, optimizer, lambda_weight, device):
    """
    Description:
        Runs one epoch of training. For each batch, computes the
        LAB and FRB losses, combines them, and updates the model.

    Input:
        model: OADN model instance
        train_loader: DataLoader for training data
        criterion: nn.CrossEntropyLoss instance
        optimizer: SGD optimizer instance
        lambda_weight: float
            Weight for combining LAB and FRB losses (Equation 5).
        device: torch.device
            'cuda' or 'cpu'

    Output:
        avg_loss: float
            Average total loss over all batches in the epoch.
        accuracy: float
            Training accuracy for the epoch (using LAB predictions).
    """
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels, attention_maps in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        attention_maps = attention_maps.to(device)

        # Forward pass
        lab_logits, frb_logits_list = model(images, attention_maps)

        # LAB loss
        lab_loss = criterion(lab_logits, labels)

        # FRB loss: sum of cross-entropy over all K regions
        frb_loss = sum(criterion(region_logits, labels) for region_logits in frb_logits_list) / len(frb_logits_list)

        # Combined loss (Equation 5)
        loss = lambda_weight * lab_loss + (1 - lambda_weight) * frb_loss

        # Backward pass and update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track metrics
        total_loss += loss.item() * images.size(0)
        _, predicted = lab_logits.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, test_loader, device):
    """
    Description:
        Evaluates the model on the test set. Averages the prediction
        scores from the LAB branch and all FRB region classifiers
        to get the final prediction, as described in Section 4.2.

    Input:
        model: OADN model instance
        test_loader: DataLoader for test data
        device: torch.device

    Output:
        accuracy: float
            Total accuracy on the test set.
        predictions: numpy array
            All predicted labels.
        ground_truths: numpy array
            All ground truth labels.
    """
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, labels, attention_maps in test_loader:
            images = images.to(device)
            attention_maps = attention_maps.to(device)

            lab_logits, frb_logits_list = model(images, attention_maps)

            # Average predictions from LAB and all FRB regions
            all_logits = [lab_logits] + frb_logits_list
            avg_logits = torch.stack(all_logits).mean(dim=0)

            _, predicted = avg_logits.max(1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    predictions = np.array(all_predictions)
    ground_truths = np.array(all_labels)
    accuracy = (predictions == ground_truths).sum() / len(ground_truths)

    return accuracy, predictions, ground_truths


def train_and_evaluate(T=0.6, grid_rows=2, grid_cols=2, lambda_weight=0.6,
                       num_epochs=60, lr=0.1, batch_size=128,
                       train_image_dir=None, test_image_dir=None,
                       train_points_path=None, test_points_path=None,
                       save_dir=None, device=None):
    """
    Description:
        Full training and evaluation pipeline. Creates the model,
        datasets, dataloaders, optimizer, and scheduler, then runs
        training for the specified number of epochs followed by
        evaluation on the test set.

    Input:
        T: float
            Confidence threshold for attention maps.
        grid_rows: int
            Number of rows in FRB grid.
        grid_cols: int
            Number of columns in FRB grid.
        lambda_weight: float
            Loss combination weight (Equation 5).
        num_epochs: int
            Total training epochs.
        lr: float
            Initial learning rate.
        batch_size: int
            Mini-batch size.
        train_image_dir: String
            Path to training images folder.
        test_image_dir: String
            Path to test images folder.
        train_points_path: String
            Path to training set interest points .pth file.
        test_points_path: String
            Path to test set interest points .pth file.
        save_dir: String
            Directory to save the trained model.
        device: torch.device
            'cuda' or 'cpu'.

    Output:
        test_accuracy: float
            Final test set accuracy.
    """
    # Transforms
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # Datasets
    train_dataset = RAFDataset(train_image_dir, train_points_path,
                               train_transform, T=T)
    test_dataset = RAFDataset(test_image_dir, test_points_path,
                              test_transform, T=T)

    # Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                             shuffle=False, num_workers=2)

    # Model
    model = OADN(num_classes=7, grid_rows=grid_rows, grid_cols=grid_cols)
    model = model.to(device)

    # Optimizer and scheduler
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=0.9, weight_decay=0.0005)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer,
                                                 step_size=20,
                                                 gamma=0.1)

    # Loss function
    criterion = nn.CrossEntropyLoss()

    # Training loop
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, lambda_weight, device
        )
        scheduler.step()

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Loss: {train_loss:.4f} "
              f"Train Acc: {train_acc:.4f} "
              f"LR: {scheduler.get_last_lr()[0]:.6f}")

    # Evaluation
    test_accuracy, predictions, ground_truths = evaluate(model, test_loader, device)
    print(f"\nTest Accuracy: {test_accuracy:.4f}")

    # Save model
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        model_name = f"oadn_T{T}_K{grid_rows*grid_cols}_lambda{lambda_weight}.pth"
        save_path = os.path.join(save_dir, model_name)
        torch.save(model.state_dict(), save_path)
        print(f"Model saved to: {save_path}")

    return test_accuracy


# Main execution
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Paths
    train_image_dir = "/kaggle/input/datasets/anaylg/oadn-dataset/OADN/RAF-DB/2/DATASET/train"
    test_image_dir = "/kaggle/input/datasets/anaylg/oadn-dataset/OADN/RAF-DB/2/DATASET/test"
    train_points_path = "/kaggle/input/datasets/anaylg/oadn-dataset/OADN/Preprocessed Training Data/train_24_interest_points.pth"
    test_points_path = "/kaggle/input/datasets/anaylg/oadn-dataset/OADN/Preprocessed Testing Data/test_24_interest_points.pth"
    save_dir = "/kaggle/working/trained_models"

    # Run with paper's best hyperparameters

    accuracy = train_and_evaluate(
        T=0.6,
        grid_rows=2,
        grid_cols=2,
        lambda_weight=0.6,
        num_epochs=60,
        lr=0.05,
        batch_size=64,
        train_image_dir=train_image_dir,
        test_image_dir=test_image_dir,
        train_points_path=train_points_path,
        test_points_path=test_points_path,
        save_dir=save_dir,
        device=device
    )


Using device: cuda
Epoch [1/60] Loss: 1.1870 Train Acc: 0.5861 LR: 0.050000
Epoch [2/60] Loss: 0.8273 Train Acc: 0.7304 LR: 0.050000
Epoch [3/60] Loss: 0.7124 Train Acc: 0.7656 LR: 0.050000
Epoch [4/60] Loss: 0.6316 Train Acc: 0.7975 LR: 0.050000
Epoch [5/60] Loss: 0.5799 Train Acc: 0.8124 LR: 0.050000
Epoch [6/60] Loss: 0.5573 Train Acc: 0.8204 LR: 0.050000
Epoch [7/60] Loss: 0.5013 Train Acc: 0.8415 LR: 0.050000
Epoch [8/60] Loss: 0.5050 Train Acc: 0.8404 LR: 0.050000
Epoch [9/60] Loss: 0.4698 Train Acc: 0.8514 LR: 0.050000
Epoch [10/60] Loss: 0.4544 Train Acc: 0.8558 LR: 0.050000
Epoch [11/60] Loss: 0.4242 Train Acc: 0.8644 LR: 0.050000
Epoch [12/60] Loss: 0.4122 Train Acc: 0.8705 LR: 0.050000
Epoch [13/60] Loss: 0.4048 Train Acc: 0.8722 LR: 0.050000
Epoch [14/60] Loss: 0.3733 Train Acc: 0.8864 LR: 0.050000
Epoch [15/60] Loss: 0.3756 Train Acc: 0.8839 LR: 0.050000
Epoch [16/60] Loss: 0.3642 Train Acc: 0.8859 LR: 0.050000
Epoch [17/60] Loss: 0.3419 Train Acc: 0.8955 LR: 0.050000
Epoc